**Content Produced by UF Signal Processing Society**

**Authors: Raul Valle & Contributors**

# Federated Learning & Privacy

Train on data you never collect: FedAvg across simulated clients (with the non-IID failure mode demonstrated, not just mentioned), and differential privacy's ε explained by actually running the attack it prevents.

## 1. Pre-requisites

[Training Dynamics](./Training_Dynamics.ipynb), [Concentration](../Intro_Math/Concentration/Concentration_Inequalities.ipynb) for the privacy-noise trade.

In [1]:
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as Fn
import matplotlib.pyplot as plt
torch.manual_seed(0); rng = np.random.default_rng(0)

---
### 🕐 Session 1 of 2 — *FedAvg & the Non-IID Problem* (~40 min)
**Goal:** average client models instead of pooling data; watch heterogeneity bite, then patch it.
**Feeds into:** Session 2 (differential privacy).

---

## 2. The Averaged Classroom

💡 **Intuition.** Hospitals/phones can't ship their data; FedAvg ships the *model*: each client trains locally a few epochs, the server averages the weights, repeat. With IID clients this tracks centralized training closely. The demon is **non-IID data**: when each client sees only its own slice of the world, local training drags each model toward its slice (*client drift*), and the average of specialists is not a generalist. Fewer local steps (more communication) is the classic dial.

In [2]:
# 4-class spiral-ish task; clients get IID or PATHOLOGICALLY split (one class each)
def make_data(n=4000):
    c = rng.integers(0, 4, n)
    ang = c*np.pi/2 + rng.uniform(0, np.pi/2.4, n)
    r = rng.uniform(0.5, 2, n)
    X = np.stack([r*np.cos(ang), r*np.sin(ang)], 1) + 0.1*rng.standard_normal((n, 2))
    return torch.tensor(X, dtype=torch.float32), torch.tensor(c)
Xa, ya = make_data()
Xtest, ytest = make_data(2000)

def new_model():
    torch.manual_seed(1)
    return nn.Sequential(nn.Linear(2, 64), nn.ReLU(), nn.Linear(64, 64), nn.ReLU(), nn.Linear(64, 4))

def fedavg(client_idx, rounds=30, local_epochs=5, lr=0.05):
    global_model = new_model()
    accs = []
    for rd in range(rounds):
        states = []
        for idx in client_idx:
            m = new_model(); m.load_state_dict(global_model.state_dict())
            opt = torch.optim.SGD(m.parameters(), lr=lr)
            for ep in range(local_epochs):
                opt.zero_grad()
                Fn.cross_entropy(m(Xa[idx]), ya[idx]).backward()
                opt.step()
            states.append(m.state_dict())
        avg = {k: torch.stack([s[k] for s in states]).mean(0) for k in states[0]}
        global_model.load_state_dict(avg)
        with torch.no_grad():
            accs.append((global_model(Xtest).argmax(1) == ytest).float().mean().item())
    return accs

perm = torch.randperm(len(Xa))
iid_clients = [perm[i::4] for i in range(4)]
noniid_clients = [torch.where(ya == c)[0] for c in range(4)]      # each client: ONE class

plt.figure(figsize=(8, 3))
for name, clients, le in [("IID clients, 5 local epochs", iid_clients, 5),
                          ("non-IID (1 class each), 5 local epochs", noniid_clients, 5),
                          ("non-IID, 1 local epoch (more comms)", noniid_clients, 1)]:
    accs = fedavg(clients, local_epochs=le)
    plt.plot(accs, label=f"{name} → final {accs[-1]:.0%}")
plt.legend(fontsize=7); plt.xlabel("communication round"); plt.ylabel("test accuracy")
plt.title("FedAvg: heterogeneity hurts; communication is the antidote")
plt.grid(True, alpha=0.3); plt.tight_layout(); plt.show()

/tmp/ipykernel_258744/1898286514.py:46: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.grid(True, alpha=0.3); plt.tight_layout(); plt.show()


---
### 🕐 Session 2 of 2 — *Differential Privacy, by Attack* (~40 min)
**Goal:** run a membership-style attack on released statistics; watch calibrated noise kill it — that's ε.
**Builds on:** Session 1.

---

## 3. What ε Actually Buys

💡 **Intuition.** Even *aggregates* leak: with an average over $n$ people and an attacker who knows everyone else, the release reveals the last person exactly. **Differential privacy** injects noise calibrated to the *sensitivity* (how much one person can move the output): the Laplace mechanism releases $f(D) + \mathrm{Lap}(\Delta f/\varepsilon)$, guaranteeing that any released value is at most $e^\varepsilon$ times more likely under your presence than your absence — your inclusion is *statistically deniable*. We don't recite that; we run the attack.

In [3]:
# the difference attack: attacker knows all salaries but Alice's; the mean is released
n_people = 100
salaries = rng.uniform(40, 160, n_people)                  # in $k
alice = salaries[0]

def release_mean(D, eps=None, sensitivity=160/n_people):
    m = D.mean()
    if eps is None: return m
    return m + rng.laplace(0, sensitivity/eps)

# exact release: Alice recovered to the cent
leak = release_mean(salaries)*n_people - salaries[1:].sum()
print(f"no privacy: attacker reconstructs Alice's salary = {leak:.4f}  (truth {alice:.4f})")

# DP release: the attack's guess distribution vs epsilon
for eps in [10, 1, 0.1]:
    guesses = np.array([release_mean(salaries, eps)*n_people - salaries[1:].sum() for _ in range(3000)])
    print(f"ε = {eps:4}: attacker's guess = {guesses.mean():6.1f} ± {guesses.std():5.1f}  "
          f"({'pinpointed' if guesses.std() < 2 else 'useful-ish' if guesses.std() < 20 else 'USELESS — spans the whole salary range'})")

no privacy: attacker reconstructs Alice's salary = 75.5560  (truth 75.5560)
ε =   10: attacker's guess =   75.5 ±  23.4  (USELESS — spans the whole salary range)
ε =    1: attacker's guess =   72.5 ± 224.9  (USELESS — spans the whole salary range)
ε =  0.1: attacker's guess =   82.0 ± 2269.5  (USELESS — spans the whole salary range)


In [4]:
# and the price: utility of the released statistic vs ε (the privacy-utility frontier)
eps_grid = np.logspace(-1.5, 1.5, 15)
err = [np.std([release_mean(salaries, e) - salaries.mean() for _ in range(2000)]) for e in eps_grid]
plt.figure(figsize=(7, 2.6))
plt.loglog(eps_grid, err, "o-")
plt.xlabel("ε (weaker privacy →)"); plt.ylabel("std of released mean [$k]")
plt.title("the privacy–utility frontier: error ∝ 1/ε — nothing is free")
plt.grid(True, which="both", alpha=0.3); plt.tight_layout(); plt.show()
print("in FL practice: DP-SGD clips each client's update (bounding sensitivity) and adds this same")
print("calibrated noise to the aggregate — Session 1's averaging with Session 2's deniability")

in FL practice: DP-SGD clips each client's update (bounding sensitivity) and adds this same
calibrated noise to the aggregate — Session 1's averaging with Session 2's deniability


/tmp/ipykernel_258744/3173407479.py:8: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.grid(True, which="both", alpha=0.3); plt.tight_layout(); plt.show()


## 4. Conclusion

FedAvg trades pooling for averaging (and pays a measured price under heterogeneity, refundable via communication); DP makes single-person influence deniable, with the attack's collapse — and the utility bill — both measured. Together they're the toolkit for learning from data nobody may hand you.

---
## Where next

- [Distributed Training II](../Intro_GPU/Distributed_Training_2.ipynb) — the same averaging, for speed instead of privacy.
- [Concentration](../Intro_Math/Concentration/Concentration_Inequalities.ipynb) — why clipped sums concentrate, which is what makes DP-SGD analyzable.